# Predicción del retraso de llegada

Este cuaderno entrena un modelo con `vuelos_asturias - vuelos_asturias.csv`, evalúa sus resultados y permite introducir los datos de un vuelo. Predice los minutos firmados de diferencia respecto a la llegada programada: **positivo** = retraso, **0** = puntual y **negativo** = adelanto.

El objetivo es `min_retraso`. No se usan como entradas las columnas que contienen el retraso real, para evitar filtrar la respuesta al modelo. El conjunto tiene 320 vuelos, así que los resultados son orientativos y mejorarán con más datos reales.

In [7]:
# %pip install pandas scikit-learn skl2onnx onnx onnxruntime

In [8]:
from pathlib import Path
import importlib
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, mean_absolute_error
import generate_onnx
importlib.reload(generate_onnx)  # Aplica los cambios del archivo sin reiniciar el kernel.

from generate_onnx import (
    build_metadata,
    create_model,
    encode_records,
    export_onnx,
    load_data,
    predict_flight,
    prepare_training_data,
    prompt_for_flight,
)

DATA_PATH = Path.cwd() / "vuelos_asturias - vuelos_asturias.csv"
if not DATA_PATH.is_file():
    raise FileNotFoundError(f"No se encuentra el CSV: {DATA_PATH}. Abre el cuaderno desde su carpeta.")

data = load_data(DATA_PATH)
metadata = build_metadata(data)
X, y = prepare_training_data(data, metadata)
print(f"Filas: {len(data)} | Variables del modelo: {X.shape[1]}")
data.head()

KeyError: 'aerolinea=Air Europa'

## Evaluación

Se reserva una parte de los vuelos para medir el error absoluto medio en minutos y la exactitud al clasificar retraso frente a no retraso. Después se volverá a entrenar con todos los registros para generar el modelo final.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)
model_eval = create_model().fit(X_train, y_train)
pred_test = model_eval.predict(X_test)
print(f"MAE: {mean_absolute_error(y_test, pred_test):.2f} minutos")
print(f"Exactitud retraso/no retraso: {accuracy_score(y_test > 0, pred_test > 0):.1%}")

## Entrenar con todos los datos y exportar a ONNX

Se crean `retraso_vuelos.onnx` y `retraso_vuelos_metadata.json`. El segundo archivo contiene el orden de las variables y los valores categóricos que debe respetar JavaScript.

In [ ]:
model = create_model().fit(X, y)
ONNX_PATH = DATA_PATH.with_name("retraso_vuelos.onnx")
model_path, metadata_path = export_onnx(model, metadata, ONNX_PATH)
print(f"Modelo: {model_path}")
print(f"Metadatos: {metadata_path}")

## Comprobar el modelo ONNX

Esta celda confirma que el archivo exportado puede cargar y predecir con ONNX Runtime. La aplicación JavaScript no necesita Python; solo requiere el `.onnx`, su JSON de metadatos y `onnxruntime-node` (`npm install onnxruntime-node`).

In [ ]:
import onnxruntime as ort

session = ort.InferenceSession(str(model_path), providers=["CPUExecutionProvider"])
onnx_input = session.get_inputs()[0].name
example = data.iloc[0].to_dict()
example_features = encode_records([example], metadata)
onnx_result = session.run(None, {onnx_input: example_features})[0].reshape(-1)[0]
python_result = predict_flight(model, metadata, example)["minutos_estimados"]
print(f"Predicción ONNX: {onnx_result:.2f} min | Python: {python_result:.2f} min")
assert np.isclose(onnx_result, python_result, atol=1e-4)

## Introducir un vuelo

Ejecuta la celda e introduce cada valor cuando se solicite. Las categorías de aerolínea y origen deben estar presentes en el CSV de entrenamiento.

In [ ]:
flight = prompt_for_flight()
prediction = predict_flight(model, metadata, flight)
print(f"Estado: {prediction['estado']}")
print(f"Diferencia estimada: {prediction['minutos_estimados']:.1f} minutos")

## Ejemplo de consumo desde JavaScript (Node.js)

Tras generar el modelo, instala `onnxruntime-node`. El vector debe construirse en el orden de `feature_order` del JSON; las variables categóricas se codifican con one-hot.

```js
const fs = require("node:fs");
const ort = require("onnxruntime-node");
const metadata = JSON.parse(fs.readFileSync("retraso_vuelos_metadata.json", "utf8"));

function minutesSinceMidnight(value) {
  const match = /^(\d{1,2}):(\d{2})$/.exec(value);
  if (!match || Number(match[1]) > 23 || Number(match[2]) > 59) {
    throw new Error(`Hora inválida: ${value}`);
  }
  return Number(match[1]) * 60 + Number(match[2]);
}

function encodeFlight(flight) {
  const date = new Date(`${flight.fecha_vuelo}T00:00:00Z`);
  if (Number.isNaN(date.getTime())) throw new Error("fecha_vuelo inválida");
  const weekday = new Intl.DateTimeFormat("en-US", {
    weekday: "long", timeZone: "UTC"
  }).format(date);
  const categorical = {
    aerolinea: flight.aerolinea.trim(),
    origen: flight.origen.trim(),
    dia_semana: weekday
  };
  const values = {};
  for (const [field, category] of Object.entries(categorical)) {
    if (!metadata.categorical_features[field].includes(category)) {
      throw new Error(`Categoría no entrenada para ${field}: ${category}`);
    }
    values[`${field}=${category}`] = 1;
  }
  Object.assign(values, {
    hora_salida_programada_min: minutesSinceMidnight(flight.hora_salida_programada),
    hora_llegada_programada_min: minutesSinceMidnight(flight.hora_llegada_programada),
    duracion_vuelo: Number(flight.duracion_vuelo),
    temperatura_origen: Number(flight.temperatura_origen),
    temperatura_destino: Number(flight.temperatura_destino),
    lluvia_origen: Number(flight.lluvia_origen),
    lluvia_destino: Number(flight.lluvia_destino),
    viento_destino: Number(flight.viento_destino),
    festivo: Number(flight.festivo)
  });
  for (const field of metadata.numeric_features) {
    if (!Number.isFinite(values[field])) throw new Error(`Valor inválido: ${field}`);
  }
  return Float32Array.from(metadata.feature_order.map(name => values[name] ?? 0));
}

async function predict(flight) {
  const session = await ort.InferenceSession.create("retraso_vuelos.onnx");
  const features = encodeFlight(flight);
  const result = await session.run({
    features: new ort.Tensor("float32", features, [1, features.length])
  });
  const minutes = Object.values(result)[0].data[0];
  return {
    minutos_estimados: minutes,
    estado: minutes > 0 ? "con retraso" : minutes < 0 ? "adelantado" : "puntual"
  };
}
```